# Preliminary

In [ ]:
%cd /root/autodl-tmp/Q-Vtree
import json
import os
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw
import pandas as pd

from InstructBLIP.evaluate import (
    extract_option_letter,
    decode_base64_image,
    run_vstar_inference_instructblip,
    evaluate_vstar_predictions,
    run_hrbench_inference_instructblip,
    evaluate_hrbench_predictions,
)

In [ ]:
%cd /root/autodl-tmp/Q-Vtree
from InstructBLIP.model import InstructBLIPWithTree

model_path = "checkpoints/instructblip-vicuna-7b"

tree_model = InstructBLIPWithTree.from_pretrained(
    model_path,
    torch_dtype=torch.float16,
    device_map="auto",
)

print("Model class:", tree_model.__class__.__name__)
print("Tree model loaded:", model_path)

# V-Star

In [ ]:
# ── 参数 ──
dataset_dir = "datasets/vstar_bench"
anno_file = "test_questions.jsonl"
sample_idx = 5

# ── 读取样本 ──
with open(os.path.join(dataset_dir, anno_file), "r") as f:
    samples = [json.loads(line) for line in f]
sample = samples[sample_idx]
img_path = os.path.join(dataset_dir, sample["image"])
question = sample["text"]
label = sample["label"]
print(f"Question: {question}")
print(f"Label:    {label}")
image = Image.open(img_path).convert("RGB")

# ── tree 推理 ──
import tempfile
tmp = tempfile.NamedTemporaryFile(suffix=".jpg", delete=False)
image.save(tmp.name)
tmp.close()

pred_text = tree_model.infer(
    image_path=tmp.name,
    question=question,
    max_new_tokens=16,
    use_tree=True,
)
os.unlink(tmp.name)

pred_option = extract_option_letter(pred_text)
print(f"Prediction:  {pred_text}")
print(f"Pred option: {pred_option}")
print(f"Correct:     {pred_option == label}")

# ── 可视化 ──
if tree_model._debug_patch_scores is not None:
    grid_h, grid_w = tree_model._get_patch_grid()

    # 1. score map (already flat [N])
    scores = tree_model._debug_patch_scores
    if isinstance(scores, torch.Tensor):
        scores = scores.detach().float().cpu()
    else:
        scores = torch.tensor(scores, dtype=torch.float32)
    full_score_map = scores.reshape(grid_h, grid_w).float()

    # 2. upsample score map 到原图尺寸
    scores_up = F.interpolate(
        full_score_map.unsqueeze(0).unsqueeze(0),
        size=(image.height, image.width),
        mode="bicubic",
        align_corners=False
    ).squeeze().cpu().numpy()
    scores_up = (scores_up - scores_up.min()) / (scores_up.max() - scores_up.min() + 1e-6)

    # 3. selected bboxes：_debug_merged_bboxes 已经是原图坐标，直接画
    orig_with_boxes = image.copy()
    draw = ImageDraw.Draw(orig_with_boxes)
    merged_bboxes = getattr(tree_model, "_debug_merged_bboxes", None)
    if merged_bboxes:
        for b in merged_bboxes:
            draw.rectangle([b[0], b[1], b[2], b[3]], outline="red", width=2)

    # 4. compact image
    compact_img = getattr(tree_model, "_debug_compact_image", None) or image

    # 5. 统计信息
    num_selected = getattr(tree_model, "_debug_num_selected_tokens", [])
    num_total    = getattr(tree_model, "_debug_num_total_tokens", [])
    n_sel = num_selected[0] if num_selected else 0
    n_tot = num_total[0]    if num_total    else 0
    ratio  = n_sel / n_tot if n_tot > 0 else 0.0

    # 6. 2×2 展示
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    axes[0][0].imshow(image)
    axes[0][0].set_title("Original")
    axes[0][0].axis("off")

    axes[0][1].imshow(image)
    axes[0][1].imshow(scores_up, cmap="jet", alpha=0.5)
    axes[0][1].set_title("Attention Score Map")
    axes[0][1].axis("off")

    axes[1][0].imshow(orig_with_boxes)
    axes[1][0].set_title(f"Selected ({n_sel}/{n_tot}, {ratio:.1%})")
    axes[1][0].axis("off")

    axes[1][1].imshow(compact_img)
    axes[1][1].set_title(f"Compact ({compact_img.size[0]}x{compact_img.size[1]})")
    axes[1][1].axis("off")

    plt.suptitle(f'Q: "{question[:80]}"  → "{pred_option}" (label: {label})', fontsize=11)
    plt.tight_layout()

    plt.savefig(
        f"vstar_vis_{sample_idx}.png",
        dpi=300,
        bbox_inches="tight",
    )

    plt.show()

In [ ]:
# base instructblip + vstar
pred_file = run_vstar_inference_instructblip(
    model=tree_model,
    model_type="base_instructblip",
    run_name="base_instructblip_vstar",
    max_samples=None,
)
evaluate_vstar_predictions(pred_file)

In [ ]:
# tree instructblip + vstar
pred_file = run_vstar_inference_instructblip(
    model=tree_model,
    model_type="tree_instructblip",
    run_name="tree_instructblip_vstar",
    max_samples=None,
)
evaluate_vstar_predictions(pred_file)